<a href="https://colab.research.google.com/github/therishiraj/Agentic-AI-workshop/blob/main/Day5_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q groq sentence-transformers numpy \
               chromadb langchain langchain-groq \
               langchain-chroma langchain-huggingface \
               langchain-community pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 755.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/2

In [2]:
from google.colab import userdata
import os

GROQ_API_KEY = userdata.get("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY   # some libraries read it from env
print("✅ Ready" if GROQ_API_KEY else "❌ Add your GROQ_API_KEY in Colab Secrets")

GROQ_MODEL = "llama-3.3-70b-versatile"   # our Groq model for the whole day

✅ Ready


In [3]:
# Our "knowledge base" — normally this comes from files. Kept inline to stay minimal.
corpus = [
    "Acme Corp offers 24 days of paid annual leave to all full-time employees.",
    "Employees may work from home up to 3 days per week with manager approval.",
    "The head office is at 42 MG Road, Bangalore, and opens at 9:00 AM daily.",
    "Acme reimburses home internet up to 1000 rupees per month for remote staff.",
    "New joiners are on probation for the first 6 months of employment.",
    "The annual company retreat is held every December in Goa.",
]

In [4]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2")   # tiny, free, ~80MB

# Each chunk becomes a 384-number vector
corpus_vectors = embedder.encode(corpus)
print("Shape:", corpus_vectors.shape)   # (6, 384) → 6 chunks, 384 numbers each

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Shape: (6, 384)


In [5]:
def cosine_similarity(a, b):
    """Angle-based similarity between two vectors. 1 = same, 0 = unrelated."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Sanity check: similar sentences score high
v1 = embedder.encode("How much leave do I get?")
v2 = embedder.encode("How many holidays are there?")
v3 = embedder.encode("Where is the office?")
print("leave vs holidays:", round(float(cosine_similarity(v1, v2)), 2))  # ~0.6+
print("leave vs office  :", round(float(cosine_similarity(v1, v3)), 2))  # low

leave vs holidays: 0.13
leave vs office  : 0.11


In [6]:
def retrieve(question, k=2):
    q_vec = embedder.encode(question)
    # score the question against every chunk
    scores = [cosine_similarity(q_vec, c_vec) for c_vec in corpus_vectors]
    # get indices of the top-k highest scores
    top_idx = np.argsort(scores)[::-1][:k]
    return [(corpus[i], round(float(scores[i]), 3)) for i in top_idx]

for chunk, score in retrieve("How many holidays do I get?"):
    print(f"{score}  |  {chunk}")

0.297  |  The annual company retreat is held every December in Goa.
0.28  |  Acme Corp offers 24 days of paid annual leave to all full-time employees.


In [8]:
from groq import Groq
groq_client = Groq(api_key=GROQ_API_KEY)

def rag_from_scratch(question, k=2):
    hits = retrieve(question, k)                       # ① RETRIEVE
    context = "\n".join(chunk for chunk, _ in hits)
    prompt = f"""Answer using ONLY the context below.  # ② AUGMENT
If it's not in the context, say "I don't have that information."

Context:
{context}

Question: {question}"""
    resp = groq_client.chat.completions.create(        # ③ GENERATE
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return resp.choices[0].message.content

print(rag_from_scratch("How many holidays do I get?"))
# → "According to the context, Acme Corp offers 24 days of paid annual leave..."

You get 24 days of paid annual leave.


In [ ]:
#session 2

In [9]:
import chromadb

chroma_client = chromadb.Client()   # in-memory; use PersistentClient to save to disk
collection = chroma_client.get_or_create_collection("acme")

# ChromaDB embeds automatically with all-MiniLM-L6-v2 — same model as Session 1!
collection.add(
    documents=corpus,
    ids=[f"doc_{i}" for i in range(len(corpus))],
)
print("Stored:", collection.count(), "chunks")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:07<00:00, 11.4MiB/s]


Stored: 6 chunks


In [10]:
results = collection.query(query_texts=["How many holidays do I get?"], n_results=2)
for doc in results["documents"][0]:
    print("-", doc)

- The annual company retreat is held every December in Goa.
- Acme Corp offers 24 days of paid annual leave to all full-time employees.


In [11]:
collection2 = chroma_client.get_or_create_collection("acme_tagged")
collection2.add(
    documents=corpus,
    ids=[f"d{i}" for i in range(len(corpus))],
    metadatas=[
        {"topic": "leave"},   {"topic": "remote"}, {"topic": "office"},
        {"topic": "remote"},  {"topic": "hr"},     {"topic": "events"},
    ],
)

# Only search within "remote"-tagged chunks
results = collection2.query(
    query_texts=["what can I claim?"],
    n_results=2,
    where={"topic": "remote"},   # 👈 metadata filter
)
print(results["documents"][0])

['Acme reimburses home internet up to 1000 rupees per month for remote staff.', 'Employees may work from home up to 3 days per week with manager approval.']


In [12]:
# PersistentClient writes to a folder so your index survives a restart
persistent = chromadb.PersistentClient(path="/content/chroma_store")
pcol = persistent.get_or_create_collection("acme_persist")
pcol.add(documents=corpus, ids=[f"p{i}" for i in range(len(corpus))])
print("Saved to /content/chroma_store — reload it later without re-embedding.")

Saved to /content/chroma_store — reload it later without re-embedding.


In [13]:
def rag_chroma(question, k=2):
    hits = collection.query(query_texts=[question], n_results=k)
    context = "\n".join(hits["documents"][0])
    prompt = f'''Answer using ONLY the context. If absent, say "I don't know."
Context:
{context}
Question: {question}'''
    resp = groq_client.chat.completions.create(
        model=GROQ_MODEL, messages=[{"role": "user", "content": prompt}], temperature=0)
    return resp.choices[0].message.content

print(rag_chroma("Can I work from home?"))

You may work from home, but it requires manager approval and is limited to up to 3 days per week.


In [14]:
#session 3

In [22]:
import langchain
print(langchain.__version__)

1.3.11


In [23]:
pip install langchain-classic

In [19]:
!pip list | grep langchain

langchain                                1.3.11
langchain-chroma                         1.1.0
langchain-classic                        1.0.8
langchain-community                      0.4.2
langchain-core                           1.4.8
langchain-groq                           1.1.3
langchain-huggingface                    1.2.2
langchain-protocol                       0.0.18
langchain-text-splitters                 1.1.2


In [25]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_classic.chains import RetrievalQA

# ① Embeddings (same local model, wrapped for LangChain)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# ② Vector store built straight from our text list
vectorstore = Chroma.from_texts(texts=corpus, embedding=embeddings)

# ③ Groq as the LLM
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# ④ One object that does retrieve → stuff into prompt → generate
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",                       # "stuff" = put all chunks in one prompt
    retriever=vectorstore.as_retriever(search_kwargs={"k": 2}),
    return_source_documents=True,             # so we can see WHICH chunks were used
)

result = qa.invoke({"query": "Where is the company?"})
print("Answer:", result["result"])
print("\nSources used:")
for doc in result["source_documents"]:
    print(" -", doc.page_content)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Answer: The head office of the company is located at 42 MG Road, Bangalore.

Sources used:
 - The head office is at 42 MG Road, Bangalore, and opens at 9:00 AM daily.
 - The head office is at 42 MG Road, Bangalore, and opens at 9:00 AM daily.
